# This jupyter notebook is prepared by "Savion J.G.".

# CAP 4611 — Assignment 5: NLP and Deep Learning
### Fake News Classification using Natural Language Processing and Neural Networks

# Section 1: Load Data and Perform Basic EDA

## Section 1.1 — Import Libraries and Perform NLTK Downloads

In [ ]:
# Standard Libraries ──────────────────────────────────────────────────────
import numpy as np                  # Numerical operations (arrays, math functions)
import pandas as pd                 # DataFrames — how we store and manipulate our dataset
import matplotlib.pyplot as plt     # Base plotting library
import seaborn as sns               # Higher-level plotting built on matplotlib (prettier charts)
import warnings
warnings.filterwarnings('ignore')   # Suppress non-critical warnings to keep output clean


# ── NLP Libraries ────────────────────────────────────────────────────────────
import nltk 
from nltk.corpus import stopwords              # Common words to remove (the, is, and, etc.)
from nltk.stem import WordNetLemmatizer       # Reduces words to their root form (running → run)
from nltk.tokenize import word_tokenize        # Splits a string into a list of individual words
import re                                      # Regular expressions — for pattern-based text cleaning
import string                                  # Contains punctuation constants

# ── Visualization ─────────────────────────────────────────────────────────────
from wordcloud import WordCloud        # Generates word cloud images from raw text

# ── Scikit-learn: Data Splitting ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split        # Splits data into train and test sets

# ── Scikit-learn: Pipeline ────────────────────────────────────────────────────
# Pipeline chains multiple steps (vectorizer → transformer → classifier) into one object
# so you can call .fit() and .predict() on the whole chain at once
from sklearn.pipeline import Pipeline

# ── Scikit-learn: Text Feature Extraction ────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer         # Converts text → word count matrix
from sklearn.feature_extraction.text import TfidfTransformer        # Converts word counts → TF-IDF weights

# ── Scikit-learn: Models ──────────────────────────────────────────────────────
from sklearn.naive_bayes import MultinomialNB           # Naive Bayes — works well with TF-IDF
from sklearn.neural_network import MLPClassifier        # Multi-Layer Perceptron — a deep neural network

# ── Scikit-learn: Evaluation Metrics ─────────────────────────────────────────
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# ── NLTK Data Downloads ───────────────────────────────────────────────────────
# These download the actual language data files NLTK needs to function.
# Only needs to run once per environment (Colab re-runs this each session).
nltk.download('stopwords')      # List of common English words to ignore
nltk.download('punkt')          # Tokenizer model — needed to split text into words
nltk.download('wordnet')        # Lexical database — needed by the lemmatizer
nltk.download('owm-1.4')        # Open Multilingual Wordnet — supports wordnet lookups
nltk.download('punkt_tab')      # Updated tokenizer data required by newer NLTK versions

print("All libraries imported successfully.")

## Section 1.2 — Read File Using `open()` and Display First 10 Items to Understand Column Separators

In [ ]:
# We read the file manually BEFORE loading it into pandas.
# This lets us see the raw formatting — specifically how columns are separated.
# Common separators are: comma (,), tab (\t), pipe (|), semicolon (;)

lines_list = []     # Empty list to store each line of the file as a string

# Open the file in read mode with UTF-8 encoding (handles special characters)
with open('news.csv', 'r', encoding='utf-8') as f:
    for line in f:
        lines_list.append(line.strip())     # .strip() removes the newline character at the end

# Print the first 10 lines so we can visually inspect the separator
# We slice [:200] on each line so the output doesn't flood the screen
for i, item in enumerate(lines_list[:10]):
    print(f"Item {i}:   {item[:200]}")
    print("-"*80)

# WHAT TO LOOK FOR IN THE OUTPUT:
# If columns are separated by large tab gaps → use sep='\t' when loading
# If columns are separated by commas → use sep=',' (the pandas default)


## Section 1.3 — Load Dataset into Pandas DataFrame and Show First 5 and Last 5 Rows

In [ ]:
# The file inspection above shows the news.csv is comma-separated.
df = pd.read_csv('news.csv', sep=',')

# shape returns (number of rows, number of columns) — quick sanity check
print(f"Dataset shape:  {df.shape}")
print(f"Columns:  {list(df.columns)}")

# .head() shows the first 5 rows — confirms the data loaded correctly
print("\n---- First 5 Rows ----")
display(df.head())

# .tail() shows the last 5 rows — confirms no corruption or truncation at the end
print("\n---- Last 5 Rows ----")
display(df.tail())

## Section 1.4 — Check for Null Values, Remove Rows with Null Values, and Verify Removal

In [ ]:
# ── Step 1: Check for nulls BEFORE removal ───────────────────────────────────
# .isnull() returns True where a cell is empty, False where it has a value
# .sum() counts the Trues per column (True = 1, False = 0)
print("Null values BEFORE removal")
print(df.isnull().sum())

# .sum().sum() gives the grand total of all nulls across the entire DataFrame
print(f"\nTotal null values:  {df.isnull().sum().sum()}")

# ── Step 2: Remove rows containing any null value ─────────────────────────────
# .dropna() removes any row that has at least one null value
# inplace=True modifies df directly instead of returning a new DataFrame
df.dropna(inplace=True)

# reset_index re-numbers rows from 0 after removals so the index has no gaps
# Without this, the index might look like: 0, 1, 5, 7... which can cause bugs later
df.reset_index(drop=True, inplace=False)

# ── Step 3: Verify nulls are gone ─────────────────────────────────────────────
print("\nNull values AFTER removal:")
print(df.isnull().sum())
print(f"Dataset shape after removing nulls: {df.shape}")


## Section 1.5 — Countplot: Number of News Articles in Each Subject

In [ ]:
plt.figure(figsize=(12,6))      # Width=12, height=6 inches

# sns.countplot counts how many rows match each unique value of 'subject'
# order= sorts bars from most to least frequent using value_counts()
# palette='viridis' is a colorblind-friendly color gradient
ax = sns.countplot(data=df, x='subject', order=df['subject'].value_counts().index, palette='viridis')

plt.title('Number of News Articles per Subject', fontsize=16, fontweight='bold')
plt.xlabel('Subject', fontsize=13)
plt.ylabel('Count', fontsize=13)
plt.xticks(rotation=45, ha='right')  # Rotate labels so they don't overlap each other

# ── Add count labels on top of each bar ───────────────────────────────────────
# ax.patches gives us each bar as a Rectangle object
# p.get_height() = the bar's value = the count
# p.get_x() + p.get_width()/2 = the horizontal center of the bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Section 1.6 — Countplot: Number of News Articles in Each Category (Fake vs. True)

In [ ]:
plt.figure(figsize=(7,5)) 

# Same concept as above but for the binary target column (0=fake, 1=true)
# This shows whether the dataset is balanced between the two classes
ax = sns.countplot(data=df, x='target', palette='Set2')

plt.title('Number of Fake vs True News Articles', fontsize=16, fontweight='bold')
plt.xlabel('Category (0 = Fake News, 1 = True News)', fontsize=13)
plt.ylabel('Count', fontsize=13)

# Replace the default numeric tick labels (0, 1) with more readable text
ax.set_xticklabels(['Fake News (0)', 'True News (1)'])

# Add count labels on top of each bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() /2., p.get_height()), ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

## Section 1.7 — Word Clouds: Most Frequent Words in Fake News vs. True News

In [ ]:
# ── Prepare one big text string for each category ────────────────────────────
# Filter rows where target == 0 (fake), grab the 'text' column,
# convert to strings (in case any cell is NaN), collect as a list,
# then join all articles into one massive single string separated by spaces
fake_text = ' '.join(df[df['target'] == 0]['text'].astype(str).tolist())
true_text = ' '.join(df[df['target'] == 1]['text'].astype(str).tolist())

# ── Generate Word Clouds ──────────────────────────────────────────────────────
# WordCloud automatically counts word frequency — bigger word = appears more often
# max_words=100 limits output to the 100 most frequent words
# colormap sets the color palette ('Reds' for fake, 'Blues' for true)
wc_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds', max_words=100).generate(true_text)
wc_true = WordCloud(width=800, height=400, background_color='black', colormap='Reds', max_words=100).generate(fake_text)

# ── Plot side by side ─────────────────────────────────────────────────────────
# plt.subplots(1, 2) = 1 row, 2 columns of subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# .imshow() renders the word cloud as an image inside the subplot
# interpolation='bilinear' smooths out any pixelation
axes[0].imshow(wc_fake, interpolation='bilinear')
axes[0].axis('off')     # Hide axis lines and ticks — not useful for image plots
axes[0].set_title('Word Cloud - Fake News (target = 0)', fontsize=14, fontweight='bold', color='red')

axes[1].imshow(wc_true, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud - True News (target = 1)', fontsize=14, fontweight='bold', color='blue')
plt.tight_layout()
plt.show()


### Section 1.7 — Observations on Word Clouds

- Fake News Word Cloud:
The fake news cloud is full of names like "Obama", "Hillary", and "Clinton" used in 
a sensational way. Words like "video" and "share" also stand out, which makes sense 
since fake news is often written to get clicks and shares rather than inform.

- True News Word Cloud:
The true news cloud looks completely different. Words like "Reuters", "said", 
"official", and "Washington" dominate. It reads like what you would expect from 
a professional news wire — neutral, sourced, and factual.

- Key Differences Observed:
The biggest difference is tone. Fake news leans on political names and emotional 
language to grab attention. True news relies on attribution and institutional 
language. You can almost tell which is which just from the vocabulary alone, which 
explains why a text classifier can do so well on this dataset.

## Section 1.8 — Create "AllText" Column by Concatenating Subject, Title, and Text

In [ ]:
# Concatenate title + text into one string per row.
# .astype(str) on each column prevents errors if any cell is not already a string.
# A space ' ' between each part ensures words don't run together at the joins.

# Example of what one row looks like after concatenation:
#   title   = "FBI probe helped by tip"
#   text    = "WASHINGTON (Reuters) — ..."
#   AllText = "FBI probe helped by tip WASHINGTON (Reuters) — ..."

# 'subject' is excluded — it directly encodes the fake/true label in this dataset
# (politicsNews=real, leftNews=fake), which would cause data leakage.
df['AllText'] = (df['title'].astype(str) + ' ' + df['text'].astype(str))

print("Alltext column created")

# Preview a truncated sample to confirm it looks right
print(f"\nSample (first 300 chars):\n{df['AllText'].iloc[0][:300]}")

# Show side-by-side to visually confirm the concatenation worked as expected
display(df[['title', 'AllText']].head(3))


## Section 1.9 — Save a Copy of the DataFrame for Later Use

In [ ]:
# .copy() creates a completely independent copy of the DataFrame.
# This matters because if you wrote df_backup = df instead,
# both variables would point to the SAME object in memory —
# any changes to df would also silently change df_backup.
# .copy() prevents that by allocating a separate copy.

# We need this backup in the Extra Credit section later,
# where we need the original columns (subject, title, text)
# that we are about to permanently drop from df below.
df_backup = df.copy()

print("Backup saved as 'df_backup'.")
print(f"Shape: {df_backup.shape}")
print(f"Columns: {list(df_backup.columns)}")

## Section 1.10 — Drop Unnecessary Columns (title, text, subject, date)

In [ ]:
# We already captured everything we need in AllText.
# Keeping title, text, subject, and date would just waste memory
# and potentially confuse the model with redundant or irrelevant columns.

# inplace=True modifies df directly — no need to write df = df.drop(...)
df.drop(columns=['title', 'text', 'subject', 'date'], inplace=True)

print("Remaining columns:", df.columns.tolist())
# Expected: ['target', 'AllText'] — target is our label, AllText is our feature
print(f"Shape: {df.shape}")
display(df.head())

## Section 1.11 — Calculate the Length of Each AllText Entry and Store in a 'length' Column

In [ ]:
# .apply(len) runs Python's built-in len() on every value in the AllText column
# len() counts the number of characters in the string
# The result is stored as a new column called 'length'
df['length'] = df['AllText'].apply(len)

print("Length column added.")

# .groupby('target') splits the DataFrame into two groups: 0=fake, 1=true
# .describe() gives count, mean, std, min, 25%, 50%, 75%, max for each group
# This helps us see if fake and true news articles tend to be different lengths
print("\nDescriptive stats by category:")
display(df.groupby('target')['length'].describe())

display(df[['AllText', 'target', 'length']].head())

## Section 1.12 — Histograms: Distribution of Text Lengths for Fake News vs. True News

In [ ]:
# Separate the lengths into two groups for individual plotting
fake_lengths = df[df['target'] == 0]['length']         # All fake article lengths
true_lengths =df[df['target'] == 1]['length']          # All true article lengths

fig, axes = plt.subplots(1, 2, figsize=(16,5))        # Side-by-side plots

# ── Fake News Histogram ───────────────────────────────────────────────────────
# bins=50 divides the full range of lengths into 50 equal-width buckets
# edgecolor='black' draws a thin border on each bar so they're visually separated
# alpha=0.8 makes bars slightly transparent — looks cleaner
axes[0].hist(fake_lengths, bins=50, color='tomato', edgecolor='black', alpha=0.8)
axes[0].set_title('Text Length - Fake News (target = 0)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character Length', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)

# axvline draws a vertical dashed line at the mean so you can instantly
# see the average article length without reading any numbers
axes[0].axvline(fake_lengths.mean(), color='darkred', linestyle='--', label=f'Mean: {fake_lengths.mean():.0f}')
axes[0].legend()

# ── True News Histogram ───────────────────────────────────────────────────────

axes[1].hist(true_lengths, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Text Length - True News (target = 1)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Character Length', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].axvline(true_lengths.mean(), color='darkblue', linestyle='--', label=f'Mean: {true_lengths.mean():.0f}')
axes[1].legend()
plt.tight_layout()
plt.show()

### Observations on Length Distribution Plots

Fake News Length Distribution:
Fake news article lengths are all over the place. Some are very short and some are 
extremely long, giving the histogram a wide, spread-out shape. This makes sense 
since fake news comes from many different types of sources with no consistent format.

True News Length Distribution:
True news articles are much more consistent in length. The histogram is tighter and 
more concentrated around the mean. This reflects the fact that most of the true news 
in this dataset comes from Reuters, which follows standard editorial length guidelines.

Key Differences or Similarities:
True news is more uniform in length while fake news is all over the place. Both 
distributions are skewed right, meaning a small number of very long articles pull 
the average up for both categories. Overall, true news tends to be slightly longer 
on average, likely because professional reporting includes more context and sourcing.

## 1.13 — Text Preprocessing Function

In [ ]:
# Initialize tools OUTSIDE the function so they're only built once,
# not re-created on every single call (much more efficient at scale)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# set() makes membership checks (word in stop_words) O(1) instead of O(n)

def preprocess_text(text):
    """
    Cleans and normalizes raw text for NLP Classification

    Steps preformed:
        1. Lowercase                - 'Trump' and 'trump' become the same word
        2. Remove URLs              - web links and noise are not meaningful
        3. Remove non-alpha chars   - strips numbers, punctuation, symbols
        4. Tokenize                 - splits string into list of individual word strings
        5. Remove Stopwords         - drops words like 'the', 'and', 'is' (no signal)
        6. Remove Short Tokens      - drops 1-2 charater leftovers (noise)
        7. Lemmatize                - reduces words to dictionary root: 'running' → 'run'

    Returns a single cleaned string of meaningful words joined by spaces. CountVectorizer(used later) expects a sting, not a list.
    """

    # Step 1 — Lowercase everything
    text = text.lower()

    # Step 2 — Remove URLs
    # \S+ matches any sequence of non-whitespace characters
    text = re.sub(r'http\S+|www\S+', '', text)

    # Step 3 — Keep only lowercase letters and whitespace
    # [^a-z\s] matches anything that is NOT a letter or space → replace with ''
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 4 — Tokenize into a list of word strings
    # word_tokenize is smarter than .split() — handles edge cases better
    tokens = word_tokenize(text)

    # Steps 5, 6, 7 — Filter and lemmatize in one pass
    # Skip stopwords, skip tokens with 2 or fewer characters, then lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]

    # Rejoin into a single string for CountVectorizer
    return ' '.join(tokens)

# ── Quick test to confirm the function works ──────────────────────────────────
sample = df['AllText'].iloc[0]
print("Original (first 200 chars):")
print(sample[:200])
print("\nPreprocessed (first 200 chars):")
print(preprocess_text(sample)[:200])


## 1.14 — Written Explanation: TF-IDF, Bag of Words, and TF-IDF Generation

### What is TF-IDF?

TF-IDF stands for Term Frequency–Inverse Document Frequency. It is a way of measuring 
how important a word is to a specific document compared to the rest of the dataset.

- Term Frequency (TF) counts how often a word appears in one document. 
  The more it appears, the higher the score.

- Inverse Document Frequency (IDF) checks how common a word is across all 
  documents. Words like "the" or "said" appear everywhere, so they get a low score. 
  Words that only appear in a few articles get a high score because they are more 
  distinctive.

- TF-IDF Score is just TF multiplied by IDF. A word scores high when it shows 
  up a lot in one document but rarely in others — meaning it actually tells you 
  something unique about that document.

### How to Create a Bag of Words Using sklearn?

A Bag of Words turns text into a table of word counts. Each row is a document and 
each column is a word from the vocabulary. The cell value is how many times that word 
appeared. In sklearn we use CountVectorizer to build this:

"from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)"

### How to Generate TF-IDF from the Bag of Words?

Once we have the word count matrix from CountVectorizer, we pass it through 
TfidfTransformer to convert the raw counts into TF-IDF weights:

"from sklearn.feature_extraction.text import TfidfTransformer
tfidf = TfidfTransformer()
X_tfidf = tfidf.fit_transform(X)"


In this assignment both steps are chained together in a Pipeline so they run 
automatically in sequence every time we train or predict.

# Section 2: Train-Test Split

## 2.1 — Perform Train-Test Split (80% Train / 20% Test)

In [ ]:
# ── Define features and labels ────────────────────────────────────────────────
# X = what the model learns from (the article text)
# y = what the model is trying to predict (0=fake, 1=true)
X = df['AllText']
y = df['target']

# ── Perform the split ─────────────────────────────────────────────────────────
# test_size=0.20 → 20% goes to test, 80% goes to training (as required)
# random_state=42 → seeds the shuffle so the split is reproducible every run
# stratify=y → preserves the fake/true class ratio in both train and test sets
#   Without stratify, by random chance you might get 70% fake in train
#   but only 50% fake in test, which would make accuracy metrics misleading
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training set size : {len(X_train)} samples")
print(f"Test set size     : {len(X_test)} samples")

# Confirm both splits have a similar fake/true ratio
print(f"\nTraining target distribution: \n{y_train.value_counts()}")
print(f"\nTest target distribution: \n{y_test.value_counts()}")

## 2.2 — Countplot: Fake vs. True Distribution in Training Set and Test Set

In [ ]:
# Build small DataFrames from the split labels so we can plot them
train_df = pd.DataFrame({'target': y_train, 'split': 'Train'})
test_df  = pd.DataFrame({'target': y_test, 'split': 'Test'})

# Combine into one DataFrame so we can filter by 'split' when plotting
split_df = pd.concat([train_df, test_df], ignore_index=True)

# Map numeric labels to readable strings for cleaner axis labels
split_df['Category'] = split_df['target'].map({0: 'Fake (0)', 1: 'True (1)'})

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ── Training set countplot ────────────────────────────────────────────────────
ax0 = sns.countplot(data=split_df[split_df['split'] == 'Train'], x = 'Category', palette = 'Set2', ax = axes[0])
axes[0].set_title('Training Set - Fake vs True News', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
for p in ax0.patches:
    ax0.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=11)

# ── Test set countplot ────────────────────────────────────────────────────────
ax1 = sns.countplot(data=split_df[split_df['split'] == 'Test'], x = 'Category', palette = 'Set1', ax = axes[1])
axes[1].set_title('Test Set - Fake vs True News', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()


# Section 3: Training and Testing a Fake News Classifier Using Multinomial Naive Bayes

## 3.1 — Build Pipeline: CountVectorizer → TfidfTransformer → MultinomialNB

In [ ]:
# Pipeline chains multiple steps into one reusable object.
# When you call pipeline.fit(X_train, y_train):
#   Step 1 runs fit_transform on X_train (learns vocab, builds count matrix)
#   Step 2 runs fit_transform on that output (learns IDF weights)
#   Step 3 runs fit on that output (trains the classifier)
# When you call pipeline.predict(X_test):
#   Step 1 runs transform only — uses the TRAINING vocabulary, no re-fitting
#   Step 2 runs transform only — uses the TRAINING IDF weights, no re-fitting
#   Step 3 runs predict
# This design prevents data leakage — test data never influences the fitting steps

nb_pipeline = Pipeline([
    # ── Step 1: CountVectorizer ───────────────────────────────────────────────
    # Converts raw text strings into a sparse matrix of word counts.
    # Each column = one unique word (or phrase) from the training vocabulary.
    # Each row = one article. Cell value = how many times that word appeared.
    ('vectorizer', CountVectorizer(
        analyzer='word',                # Analyze at the word level (not character level)
        preprocessor=preprocess_text,   # Run our custom cleaning function before tokenizing
        ngram_range=(1,2),              # Capture single words AND two-word phrases
        min_df=2                        # e.g., both "fake" and "fake news" become features
                                        # Ignore words appearing in fewer than 2 documents
                                        # removes rare typos and one-off noise
    )),

    # ── Step 2: TfidfTransformer ──────────────────────────────────────────────
    # Converts raw word counts from Step 1 into TF-IDF weights.
    # Words that appear in nearly every document (like "said") get down-weighted.
    # Words that are common in only a few documents get up-weighted (more distinctive).
    # Result: features that carry more meaningful signal for classification.
    ('tfidf', TfidfTransformer()),

     # ── Step 3: MultinomialNB ─────────────────────────────────────────────────
    # Naive Bayes classifier designed for discrete count/frequency data like TF-IDF.
    # "Naive" = it assumes all word features are statistically independent of each other.
    # Despite that unrealistic assumption, it performs very well on text classification.
    # It's extremely fast to train and is a strong baseline model.
    ('Classifier', MultinomialNB())
])

print("Naive Bayes pipeline created:")
print(nb_pipeline)

## 3.2 — Fit the Pipeline and Perform Predictions

In [ ]:
print("Training Naive Bayer Pipeline...")

# .fit() triggers the entire pipeline on the training data in sequence:
#   CountVectorizer builds its vocabulary and creates the word count matrix
#   TfidfTransformer learns the IDF weights from the training corpus
#   MultinomialNB learns the probability of each word appearing per class (fake/true)
nb_pipeline.fit(X_train, y_train)
print("Training Complete.")

# .predict() runs X_test through the TRAINED pipeline:
#   CountVectorizer maps test text to the training vocabulary (no new learning)
#   TfidfTransformer applies the training IDF weights (no new learning)
#   MultinomialNB outputs 0 (fake) or 1 (true) for each article
y_pred_nb = nb_pipeline.predict(X_test)

print(f"Predications generated for {len(y_pred_nb)} test samples.")


## 3.3 — Classification Report and Confusion Matrix (Naive Bayes)

In [ ]:
print("="*55)
print(" NAIVE BAYES - Classification Report")
print("="*55)

# classification_report compares y_test (true labels) to y_pred_nb (predicted labels)
# For each class it shows:
#   Precision  = of all articles predicted as class X, what fraction actually were X?
#   Recall     = of all actual class X articles, what fraction did we correctly catch?
#   F1-score   = harmonic mean of precision and recall — single balanced metric
#   Support    = number of true instances of that class in the test set
#   Accuracy   = fraction of ALL predictions that were correct
print(classification_report(y_test, y_pred_nb, target_names=['Fake News (0)', 'True News (1)']))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
# A 2x2 grid layout:
#   [True Negative  | False Positive]   TN = predicted fake,  actually fake  ✓
#   [False Negative | True Positive ]   TP = predicted true,  actually true  ✓
#                                       FP = predicted true,  actually fake  ✗
#                                       FN = predicted fake,  actually true  ✗
cm_nb = confusion_matrix(y_test, y_pred_nb)

# ConfusionMatrixDisplay wraps the matrix for clean heatmap visualization
disp = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=['Fake (0)', 'True (1)'])
fig, ax = plt.subplots(figsize=(7,5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix - Multinomial Naive Bayes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



## 3.4 — Discussion: Naive Bayes Model Performance

- Overall Performance:
The Naive Bayes model hit 96% accuracy on 8,980 test articles, which meets the 
assignment requirement. What stands out is how fast it got there — it trained in 
seconds. For a simple probabilistic model, that is a genuinely impressive result.

- Performance on Fake News (class 0): 
Precision was 0.97 and recall was 0.95, giving an F1 of 0.96. When the model called 
something fake, it was right 97% of the time. It missed about 5% of actual fake 
articles, meaning a small number slipped through as predicted true.

- Performance on True News (class 1): 
Precision was 0.95 and recall was 0.97, also F1 of 0.96. The model caught 97% of 
all real news correctly. The slightly lower precision means a handful of fake articles 
were incorrectly called true.

- Strengths and Weaknesses:
The biggest strength of Naive Bayes is speed — it is nearly instant to train and 
works surprisingly well for something this simple. The weakness is that it treats 
every word as independent from every other word, which is not how language actually 
works. That assumption is what holds it back from the accuracy the neural network 
achieves.

## 3.5 — Predict a Real-World News Article Using the Naive Bayes Model

In [ ]:
# Paste any news article excerpt between the triple quotes below.
# Copy a paragraph or two from CNN, Reuters, BBC, etc.
real_world_article = """
The United States and China agreed to temporarily reduce tariffs on each other's 
goods following trade talks held in Geneva, offering relief to businesses and 
consumers affected by months of escalating trade tensions. The agreement, 
announced by U.S. Treasury Secretary Scott Bessent, will see the United States 
lower tariffs on Chinese imports from 145 percent to 30 percent for a 90 day 
period while China will reduce its tariffs on American goods from 125 percent 
to 10 percent. Officials from both sides described the talks as productive and 
said they had established a framework for continued negotiations. The deal 
marks a significant de-escalation in a trade dispute that had rattled global 
markets and disrupted supply chains across multiple industries. Economists 
welcomed the pause but cautioned that a permanent resolution would require 
further rounds of negotiation on deeper structural issues between the two 
largest economies in the world.
"""

# Pass the article as a list with one element — predict() always expects an iterable
prediction_nb = nb_pipeline.predict([real_world_article])

# prediction_nb[0] is either 0 (fake) or 1 (true)
# We map it to a readable label for display
label_nb = 'True News' if prediction_nb[0] == 1 else 'Fake News'

print(f"Naive Bayes Prediction: {prediction_nb[0]}")
print(f"The model classified this article as: {label_nb}")


# Section 4: Training and Testing a Deep Neural Network (MLPClassifier)

## 4.1 — Import MLPClassifier from sklearn.neural_network

In [ ]:
# MLPClassifier = Multi-Layer Perceptron Classifier
# This is sklearn's feedforward deep neural network implementation.
# Already imported at the top of the notebook in Section 1.1,
# but the assignment asks us to show it explicitly here.
from sklearn.neural_network import MLPClassifier

print("MLPClassifier imported successfully")

## 4.2 — Build Pipeline: CountVectorizer → TfidfTransformer → MLPClassifier (2 hidden layers, verbose=2)

In [ ]:
mlp_pipeline = Pipeline([

    # ── Step 1: CountVectorizer ───────────────────────────────────────────────
    # Converts raw text strings into a sparse matrix of word counts.
    # max_features=20000 caps the vocabulary so the matrix doesn't get too large
    # min_df=10 drops any word that appears in fewer than 10 articles (noise removal)
    # ngram_range=(1,1) uses single words only — bigrams doubled the matrix size
    ('vectorizer', CountVectorizer(
        analyzer='word',
        preprocessor=preprocess_text,
        ngram_range=(1, 1),    # Unigrams only — bigrams were too heavy for local machine
        min_df=10,             # Ignore words appearing in fewer than 10 documents
        max_features=20000     # Hard cap on vocabulary size — keeps matrix manageable
    )),

    # ── Step 2: TfidfTransformer ──────────────────────────────────────────────
    # Converts raw word counts into TF-IDF weights.
    # Down-weights common words, up-weights distinctive ones.
    ('tfidf', TfidfTransformer()),

    # ── Step 3: MLPClassifier ─────────────────────────────────────────────────
    # Two hidden layers as required by the assignment.
    # Smaller than originally planned but still more than enough for this dataset.
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        # Two hidden layers:
        #   128 neurons in layer 1 — extracts broad patterns from TF-IDF features
        #   64 neurons in layer 2  — refines those into higher-level representations
        # Output layer (2 neurons for fake/true) is added automatically by sklearn

        activation='relu',
        # ReLU: f(x) = max(0, x) — standard activation for modern neural networks
        # Prevents vanishing gradient problem, trains faster than sigmoid

        solver='adam',
        # Adaptive optimizer — adjusts learning rate per parameter automatically
        # Best general-purpose choice for this type of problem

        max_iter=10,
        # 10 passes through the training data — enough for this dataset to converge
        # Reduces training time significantly vs the original 20

        random_state=42,
        # Seeds weight initialization so results are reproducible

        verbose=2
        # Prints loss after every iteration so you can watch the model learn
        # Loss should decrease each iteration — when it flattens, it has converged
    ))
])

print("MLP Neural Network pipeline created:")
print(mlp_pipeline)

## 4.3 — Fit the Pipeline and Perform Predictions

In [ ]:
print("Training MLP Neural Network (should take 2-4 minutes)...")

# .fit() runs all three pipeline steps on the training data:
#   CountVectorizer builds the vocabulary and creates the word count matrix
#   TfidfTransformer learns the IDF weights from the training corpus
#   MLPClassifier trains the neural network via backpropagation
# Watch the loss values — they should decrease each iteration
mlp_pipeline.fit(X_train, y_train)
print("\nTraining complete.")

# .predict() applies all trained steps to the test set
# Returns 0 (fake) or 1 (true) for each article
y_pred_mlp = mlp_pipeline.predict(X_test)

print(f"Predictions generated for {len(y_pred_mlp)} test samples.")

## 4.4 — Classification Report and Confusion Matrix (MLP Neural Network)

In [ ]:
print("="*55)
print("  MLP NEURAL NETWORK - Classification Report")
print("="*55)

# Same report format as Naive Bayes — makes direct comparison between models easy.
# Target: ≥99% accuracy to receive full credit for this section.
print(classification_report(y_test, y_pred_mlp, target_names=['Fake News (0)', 'True News (1)']))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
# A well-performing model has large numbers on the diagonal (correct predictions)
# and small numbers off the diagonal (misclassifications).
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
disp_mlp = ConfusionMatrixDisplay(confusion_matrix=cm_mlp, display_labels=['Fake News (0)', 'True News (1)'])
fig, ax = plt.subplots(figsize=(7,5))
disp_mlp.plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Confusion Matrix - MLP Neural Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.5 — Discussion: MLP Neural Network Performance

- Overall Performance:
The MLP hit 100% accuracy on the test set, which exceeds the 99% requirement. 
Watching the loss drop from 0.157 in the first iteration all the way down to 0.0004 
by iteration 10 showed how quickly the network learned the patterns in the data.

- Performance on Fake News (class 0):
Both precision and recall came in at 1.00 for fake news — a perfect score. The model 
did not misclassify a single fake article on the test set, which is remarkable 
considering the test set had 4,696 fake articles.

- Performance on True News (class 1):
Precision was 1.00 and recall was 0.99 for true news, giving an F1 of 1.00 rounded. 
Less than 1% of true articles were incorrectly labeled fake. In a dataset this size 
that is a negligible error rate.

- Strengths and Weaknesses:
The MLP clearly outperforms Naive Bayes by learning patterns in the data that a 
simple probability model cannot detect. The tradeoff is training time — even the 
optimized version took several minutes on a local machine. For a one-time training 
job that is fine, but it would be something to consider in a system that retrains 
frequently on new data.

## 4.6 — Predict the Same Real-World News Article Using the MLP Model

In [ ]:
# Uses the same real_world_article string defined in Section 3, Question 3.5
# No need to paste the text again — it's still stored in memory
prediction_mlp = mlp_pipeline.predict([real_world_article])
label_mlp = 'True News' if prediction_mlp[0] == 1 else 'Fake News'

print(f"MLP Prediction: {prediction_mlp[0]}")
print(f"The model classified this article as: {label_mlp}")

# ── Side-by-side comparison ───────────────────────────────────────────────────
# Useful for the discussion in Section 4, Question 4.7
# Do both models agree? If not, which do you trust more and why?
print("\n" + "=" *50)
print(" Side-by-Side Model Comparsion")
print("="*55)
print(f"    Naive Bayes     : {'True News' if prediction_nb[0] == 1 else 'Fake News'}")
print(f"    MLP Network     : {label_mlp}")
print("="*55)


## 4.7 — Discussion: Comparing MLP vs. Naive Bayes

- Accuracy Comparison:
MLP at 100% vs Naive Bayes at 96% — the neural network wins by a clear margin. 
That 4% gap might sound small but it translates to roughly 360 fewer misclassified 
articles in a test set of 8,980. In a real application that difference matters.

- Precision / Recall / F1 Comparison:
Both models scored F1 of 0.96 for Naive Bayes vs effectively 1.00 for MLP across 
both classes. The MLP made almost no errors in either direction while Naive Bayes 
had a consistent small error rate on both fake and true articles.

- Speed and Complexity Trade-offs:
Naive Bayes trains in under a second. The MLP took several minutes even after 
optimizing the settings for a local machine. Once trained though, both predict at 
the same speed. The extra training time is a one-time cost.

- Which model would you recommend and why?
For this task I would recommend the MLP. The accuracy difference is significant 
enough to justify the longer training time, especially in a fake news detection 
context where a wrong classification has real consequences. That said, if you needed 
something fast to prototype or retrain daily on new articles, Naive Bayes is still 
a solid choice that gets you most of the way there.

## EC.1 — Prepare the DataFrame for Subject Classification

In [ ]:
# Pull from the backup saved in Section 1, Question 1.9.
# We need the original columns (subject, title, text) that were dropped from df.
df_subject = df_backup.copy()

# Rebuild AllText the same way as Section 1, Question 1.8
df_subject['AllText'] = (df_subject['subject'].astype(str) + ' ' + df_subject['title'].astype(str) + ' ' + df_subject['text'].astype(str))

# Drop columns we don't need for this task.
# KEY DIFFERENCE from earlier: we DROP 'target' (fake/true) because we're NOT
# predicting fake vs. true here — we're predicting which SUBJECT the article belongs to.
# We KEEP 'subject' because that's our new target label.
df_subject.drop(columns=['title', 'text', 'date', 'target'], inplace=True)

print("Subject Classification DataFrame prepared.")
print(f"Shape; {df_subject.shape}")
print(f"Columns: {list(df_subject.columns)}")       # Should be: ['subject', 'AllText']

# View class distribution — subjects with fewer articles may be harder to classify
print("\nSubject value counts:")
print(df_subject['subject'].value_counts())



## EC.2 — Train-Test Split for Subject Classification

In [ ]:
X_sub = df_subject['AllText']       # Features: full article text
y_sub = df_subject['subject']       # Target: subject name (multi-class, not binary)

# Same 80/20 split strategy as Section 2.
# Stratify is especially important here because some subjects have far fewer
# articles than others — without it, rare subjects might not appear in the test set
X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(X_sub, y_sub, test_size=0.20, random_state=42, stratify=y_sub)

print(f"Training set    : {len(X_sub_train)} samples")
print(f"Test set        : {len(X_sub_test)} samples")

# Confirm every subject class appears in the training set
print(f"Class distribution in training set: \n{y_sub_train.value_counts()}")

## EC.3 — Build, Train, and Evaluate MLP Pipeline for Subject Classification

In [ ]:
subject_pipeline = Pipeline([

    # ── Step 1: CountVectorizer ───────────────────────────────────────────────
    # Same optimizations as the main MLP pipeline —
    # unigrams only, higher min_df, and a capped vocabulary
    # to keep the feature matrix manageable on a local machine
    ('vectorizer', CountVectorizer(
        analyzer='word',
        preprocessor=preprocess_text,
        ngram_range=(1, 1),    # Unigrams only — bigrams too heavy for local machine
        min_df=10,             # Ignore words appearing in fewer than 10 documents
        max_features=20000     # Hard cap on vocabulary size
    )),

    # ── Step 2: TfidfTransformer ──────────────────────────────────────────────
    # Converts raw word counts into TF-IDF weights.
    # Down-weights common words, up-weights distinctive ones.
    ('tfidf', TfidfTransformer()),

    # ── Step 3: MLPClassifier for multi-class subject prediction ─────────────
    # Multi-class is harder than binary so we use a slightly larger network
    # than the fake/true classifier, but still kept small enough to run locally.
    # Three hidden layers give the model more capacity to distinguish subjects.
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        # Three hidden layers:
        #   256 → broad pattern detection across all subject categories
        #   128 → intermediate feature refinement
        #   64  → compact representation fed to the output layer
        # Output layer has one neuron per unique subject — added automatically

        activation='relu',
        # ReLU: f(x) = max(0, x) — standard for modern neural networks
        # Prevents vanishing gradient, trains faster than sigmoid

        solver='adam',
        # Adaptive optimizer — handles sparse TF-IDF data well
        # Adjusts learning rate per parameter automatically

        max_iter=10,
        # 10 iterations — enough for this dataset to converge
        # Keeps training time reasonable on a local machine

        random_state=42,
        # Seeds weight initialization so results are reproducible

        verbose=2
        # Prints loss after every iteration so you can watch the model learn
    ))
])

print("Training Subject Classification pipeline (should take 3-5 minutes)...")

# sklearn handles multi-class automatically — no extra config needed.
# The output layer will have one neuron per unique subject label.
subject_pipeline.fit(X_sub_train, y_sub_train)
print("\nTraining complete.")

y_sub_pred = subject_pipeline.predict(X_sub_test)
print(f"Predictions generated for {len(y_sub_pred)} test samples.")

## EC.4 — Classification Report and Confusion Matrix for Subject Classification

In [ ]:
print("="*60)
print(" SUBJECT CLASSIFIER - Classification Report")
print("="*60)

# Report now shows one row per subject instead of just fake/true.
# Look for subjects with low recall — those are the classes the model struggles with.
# Subjects with fewer training samples typically have lower scores.
print(classification_report(y_sub_test, y_sub_pred))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
# For multi-class, the matrix is NxN where N = number of unique subjects.
# Diagonal = correct predictions per subject.
# Off-diagonal = which subjects get confused for each other.
# For example, if 'politicsNews' and 'politics' show high off-diagonal values,
# those two subjects share so much vocabulary the model can't separate them.

# subject_pipeline.classes_ gives the sorted list of unique subject labels
# We pass it to labels= so matrix rows and columns are correctly labeled
cm_sub = confusion_matrix(y_sub_test, y_sub_pred, labels=subject_pipeline.classes_)

fig,ax = plt.subplots(figsize=(12,9))
disp_sub = ConfusionMatrixDisplay(confusion_matrix=cm_sub, display_labels=subject_pipeline.classes_)

# xticks_rotation=45 tilts the x-axis labels so they don't overlap
disp_sub.plot(ax=ax, cmap='Purples', colorbar=False, xticks_rotation=45)
ax.set_title('Confusion Matrix - Subject Classification (MLP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## EC.5 — Discussion: Subject Classification Model Performance

 -Overall Accuracy:
The model achieved 93% accuracy across 8 subject categories, which comfortably 
clears the 79% threshold for full credit. Considering this is a multi-class problem 
with some categories having very few training examples, 93% is a strong result.

- Subjects the Model Classifies Well (Strengths):
The model was nearly perfect on worldnews (F1 0.98), politicsNews (F1 0.98), and 
News (F1 0.98). These three categories have the most training data — over 7,000 
samples each — which gives the model enough examples to learn clear, distinct 
vocabulary patterns for each one.

- Subjects the Model Struggles With (Weaknesses):
Government News was the weakest category by far with an F1 of only 0.58 and a 
recall of 0.53. That means the model missed nearly half of all Government News 
articles. US_News (F1 0.83) and Middle-east (F1 0.83) also underperformed compared 
to the top categories.

- Possible Reasons for Misclassifications:
Two things explain most of the errors. First, class size — Government News only had 
1,256 training samples compared to over 9,000 for politicsNews. The model simply 
did not see enough Government News articles to learn what makes it different. 
Second, topic overlap — Government News, politics, politicsNews, and US_News all 
cover American political topics and share a lot of the same vocabulary. When 
categories sound this similar, even a strong model will confuse them.